In [1]:
from dotenv import load_dotenv, find_dotenv
import os

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from typing import Union, List, Literal


load_dotenv(find_dotenv(), override=True)

tc = TavilyClient()


In [23]:
class ValidJob(BaseModel):
    """LinkedIn Job Params"""
    link: str = Field(description="HTTPS link to job posting")
    title: str = Field(description="Formal position title")
    listingDate: str = Field(description="Date of job listing in MM/DD/YYYY")
    companyName: str = Field(description="Name of company")
    location: str = Field(description="Geographic region")
    workplaceType: Literal["Remote", "Hybrid", "On-site", "Unknown"] = Field(
        "Unknown", 
        description="Work environment type"
    )
    skillsRequired: List[str] = Field(
        default_factory=list, 
        description="Top 3-5 core technical skills, languages, or tools explicit in the text."
    )

class JobList(BaseModel):
    queryType: Literal["Job"]
    numJobs: int = Field(descriptoin="Number of jobs that agent was able to find and return")
    jobs: List[ValidJob] = Field(description="List of jobs that the agent was able to find and return")

class GenQueryRes(BaseModel):
    queryType: Literal["Any"]
    answer: str = Field(description="Answer to user's query. Not for jobs")
    

/tmp/ipykernel_83402/3389649298.py:19: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptoin'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  numJobs: int = Field(descriptoin="Number of jobs that agent was able to find and return")


In [24]:

class ResObj(BaseModel):
    query: str = Field(description="Original job search query for user")
    timeQ: str = Field(description="Time of query formatted in UTC")
    answer: Union[GenQueryRes, JobList] = Field(..., discriminator="queryType")
    

In [25]:
@tool
def search(query: str) -> str:
    """
    You are a search tool that can search and return queries over the internet
    
    Input: 
        query: the query to search for
    
    Output:
        the search results
    """
    print(f"searching for {query} \n")
    return tc.search(query=query)

@tool
def validate(query: str) -> str:
    """
    You are a validation tool that can search for a query and ensure that it fits constraints
    
    Input: 
        query: the query to search for
    
    Output:
        
    """
    print(f"validating for {query} \n")
    return tc.search(query=query)

In [36]:
llm = ChatOpenAI(temperature=0, model='gpt-5.4-nano')
tools = [search, validate]
agent = create_agent(model=llm, tools=tools, response_format=ResObj)

In [37]:
print("Hello World")
result = agent.invoke({'messages': HumanMessage("give me 5 linkedin job postings for an AI developer with langchain in the bay area. Ensure that the listings are still open.")})
print(result)

Hello World
searching for site:linkedin.com/jobs AI developer LangChain Bay Area 

searching for site:linkedin.com/jobs "LangChain" "Bay Area" AI developer 

searching for site:linkedin.com/jobs/view "langchain" "San Francisco" AI Engineer 

searching for site:linkedin.com/jobs/view "langchain" "AI" "San Francisco Bay Area" "Machine Learning" "Engineer" 

searching for site:linkedin.com/jobs/view "LangChain" "Remote" "Bay Area" AI engineer 

{'messages': [HumanMessage(content='give me 5 linkedin job postings for an AI developer with langchain in the bay area. Ensure that the listings are still open.', additional_kwargs={}, response_metadata={}, id='64297c4d-5bb2-4f65-8acf-a29a207d7488'), AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 695, 'total_tokens': 721, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction

In [38]:
result['messages']

[HumanMessage(content='give me 5 linkedin job postings for an AI developer with langchain in the bay area. Ensure that the listings are still open.', additional_kwargs={}, response_metadata={}, id='64297c4d-5bb2-4f65-8acf-a29a207d7488'),
 AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 695, 'total_tokens': 721, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Dtxku7hHX0mxvm4ywOhNBK1CfTJRP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef531-52c7-7393-9e6c-cc6c21c3043b-0', tool_calls=[{'name': 'search', 'args': {'query': 'site:linkedin.com/jobs AI developer LangChain Bay Ar